# Unify IDs - Notebook for running the unifying scripts

Run the cell below to get started. Before launching this notebook, execute:

```bash
cd <path-to-your-clone>/awg-utils/unify_tkk_ids
source .venv/bin/activate  # activate virtual environment
jupyter notebook simple_unify_notebook.ipynb
```

Then return here and use the box to enter paths.

## Enter input files
add the paths underneath the following cell:

In [1]:
import os
import sys
import subprocess
import ipywidgets as widgets
from IPython.display import display

json_path = ''
svg_folder = ''

json_path_text = widgets.Text(
    description='JSON:',
    placeholder='Path to .json file',
    layout=widgets.Layout(width='62%')
)
svg_folder_text = widgets.Text(
    description='SVG:',
    placeholder='Path to SVG folder',
    layout=widgets.Layout(width='62%')
)

browse_json_button = widgets.Button(description='Browse JSON', button_style='info', layout=widgets.Layout(width='15%'))
paste_json_button = widgets.Button(description='Paste from Clipboard', button_style='warning')
browse_svg_button = widgets.Button(description='Browse SVG', button_style='info', layout=widgets.Layout(width='15%'))
paste_svg_button = widgets.Button(description='Paste from Clipboard', button_style='warning')


def _run_command(args):
    try:
        result = subprocess.run(args, capture_output=True, text=True)
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        pass
    return ''


def _clipboard_text():
    commands = []
    if sys.platform == 'darwin':
        commands.append(['pbpaste'])
    elif sys.platform.startswith('linux'):
        commands.extend([
            ['wl-paste', '-n'],
            ['xclip', '-selection', 'clipboard', '-o'],
            ['xsel', '--clipboard', '--output']
        ])
    elif os.name == 'nt':
        commands.append(['powershell', '-NoProfile', '-Command', 'Get-Clipboard'])

    commands.extend([
        ['pbpaste'],
        ['wl-paste', '-n'],
        ['xclip', '-selection', 'clipboard', '-o'],
        ['xsel', '--clipboard', '--output'],
        ['powershell', '-NoProfile', '-Command', 'Get-Clipboard']
    ])

    for command in commands:
        try:
            text = subprocess.check_output(command, text=True).strip()
            if text:
                return text
        except Exception:
            continue
    return ''


def _choose_file_json():
    if sys.platform == 'darwin':
        script = 'POSIX path of (choose file with prompt "Select JSON file")'
        path = _run_command(['osascript', '-e', script])
        return path if path.endswith('.json') else ''

    if sys.platform.startswith('linux'):
        path = _run_command(['zenity', '--file-selection', '--title', 'Select JSON file', '--file-filter=*.json'])
        if path:
            return path
        return _run_command(['kdialog', '--getopenfilename', os.getcwd(), '*.json'])

    if os.name == 'nt':
        ps = (
            "Add-Type -AssemblyName System.Windows.Forms;"
            "$d=New-Object System.Windows.Forms.OpenFileDialog;"
            "$d.Filter='JSON files (*.json)|*.json|All files (*.*)|*.*';"
            "if($d.ShowDialog() -eq [System.Windows.Forms.DialogResult]::OK){$d.FileName}"
        )
        return _run_command(['powershell', '-NoProfile', '-Command', ps])

    return ''


def _choose_folder():
    if sys.platform == 'darwin':
        script = 'POSIX path of (choose folder with prompt "Select SVG folder")'
        return _run_command(['osascript', '-e', script])

    if sys.platform.startswith('linux'):
        path = _run_command(['zenity', '--file-selection', '--directory', '--title', 'Select SVG folder'])
        if path:
            return path
        return _run_command(['kdialog', '--getexistingdirectory', os.getcwd()])

    if os.name == 'nt':
        ps = (
            "Add-Type -AssemblyName System.Windows.Forms;"
            "$d=New-Object System.Windows.Forms.FolderBrowserDialog;"
            "if($d.ShowDialog() -eq [System.Windows.Forms.DialogResult]::OK){$d.SelectedPath}"
        )
        return _run_command(['powershell', '-NoProfile', '-Command', ps])

    return ''


def _sync_paths(_=None):
    global json_path, svg_folder
    json_path = json_path_text.value.strip()
    svg_folder = svg_folder_text.value.strip()


def on_browse_json(_):
    path = _choose_file_json()
    if path:
        json_path_text.value = os.path.abspath(path)


def on_paste_json(_):
    path = _clipboard_text()
    if path:
        json_path_text.value = path


def on_browse_svg(_):
    path = _choose_folder()
    if path:
        svg_folder_text.value = os.path.abspath(path)


def on_paste_svg(_):
    path = _clipboard_text()
    if path:
        svg_folder_text.value = path


json_path_text.observe(_sync_paths, names='value')
svg_folder_text.observe(_sync_paths, names='value')
browse_json_button.on_click(on_browse_json)
paste_json_button.on_click(on_paste_json)
browse_svg_button.on_click(on_browse_svg)
paste_svg_button.on_click(on_paste_svg)

display(widgets.HBox([json_path_text, browse_json_button, paste_json_button]))
display(widgets.HBox([svg_folder_text, browse_svg_button, paste_svg_button]))
_sync_paths()

## Run TKK Unifier

In [2]:
import os
from unify_tkk_ids import unify_tkk_ids

# Validate paths
if not json_path or not os.path.isfile(json_path):
    print("❌ Error: No valid JSON file selected")
elif not svg_folder or not os.path.isdir(svg_folder):
    print("❌ Error: No valid SVG folder selected")
else:
    print(f'Running TKK Unifier...')
    print(f'  JSON: {json_path}')
    print(f'  SVG Folder: {svg_folder}')
    result = unify_tkk_ids(json_path, svg_folder)
    print(f'\n✓ Finished, result = {result}')

Running TKK Unifier...
  JSON: /Users/Elias/awg-utils/unify_tkk_ids/tests/data/textcritics.json
  SVG Folder: /Users/Elias/awg-utils/unify_tkk_ids/tests/img
--- Starting TKK ID processing ---
Loaded JSON with 25 entries
Found 30 SVG files in folder

Processing Entry ID: op25_WE
 Standard anchor: op25_WE
 Relevant SVGs (0): []
 No svgGroupIds to process

Processing Entry ID: M_317_TF1
 Standard anchor: M_317_TF1
 Relevant SVGs (2): ['M317_Textfassung1-1von2-final.svg', 'M317_Textfassung1-2von2-final.svg']
 [JSON] Changing 'g-tkk-m_317_tf1-1' -> 'g-tkk-m_317_tf1-1'
 [SVG] Changing 'g-tkk-m_317_tf1-1' -> 'g-tkk-m_317_tf1-1' in M317_Textfassung1-1von2-final.svg
 [JSON] Changing 'g-tkk-m_317_tf1-2' -> 'g-tkk-m_317_tf1-2'
 [SVG] Changing 'g-tkk-m_317_tf1-2' -> 'g-tkk-m_317_tf1-2' in M317_Textfassung1-1von2-final.svg
 [JSON] Changing 'g-tkk-m_317_tf1-3' -> 'g-tkk-m_317_tf1-3'
 [SVG] Changing 'g-tkk-m_317_tf1-3' -> 'g-tkk-m_317_tf1-3' in M317_Textfassung1-1von2-final.svg

Processing Entry ID: 

## Run LinkBox Unifier
Unify LinkBox svgGroupIds with format `{entry_id}to{sheetId}`

In [3]:
import os
import sys

# Reload modules to get the latest changes
if 'unify_link_box_ids' in sys.modules:
    del sys.modules['unify_link_box_ids']
if 'utils.extraction_utils' in sys.modules:
    del sys.modules['utils.extraction_utils']
if 'utils.svg_utils' in sys.modules:
    del sys.modules['utils.svg_utils']

from unify_link_box_ids import unify_link_box_ids

# Validate paths
if not json_path or not os.path.isfile(json_path):
    print("❌ Error: No valid JSON file selected")
elif not svg_folder or not os.path.isdir(svg_folder):
    print("❌ Error: No valid SVG folder selected")
else:
    print(f'Running LinkBox Unifier...')
    print(f'  JSON: {json_path}')
    print(f'  SVG Folder: {svg_folder}')
    result = unify_link_box_ids(json_path, svg_folder)
    print(f'\n✓ Finished, result = {result}')

Running LinkBox Unifier...
  JSON: /Users/Elias/awg-utils/unify_tkk_ids/tests/data/textcritics.json
  SVG Folder: /Users/Elias/awg-utils/unify_tkk_ids/tests/img
--- Starting Link Box ID processing ---
Loaded JSON with 25 entries
Found 30 SVG files in folder

Processing Entry ID: op25_WE
 Standard anchor: op25_WE
 Relevant SVGs (0): []
 No linkBoxes to process

Processing Entry ID: M_317_TF1
 Standard anchor: M_317_TF1
 Relevant SVGs (2): ['M317_Textfassung1-1von2-final.svg', 'M317_Textfassung1-2von2-final.svg']
 No linkBoxes to process

Processing Entry ID: M_317_Sk1
 Standard anchor: M_317_Sk1
 Relevant SVGs (1): ['M317_Sk1-1von1-final.svg']
 Found 9 linkBox(es)
 [JSON] Changing 'M_317_Sk1toM_317_Sk2' -> 'M_317_Sk1toM_317_Sk2'
 [SVG] Changing 'M_317_Sk1toM_317_Sk2' -> 'M_317_Sk1toM_317_Sk2' in M317_Sk1-1von1-final.svg
 [JSON] Changing 'M_317_Sk1toM_317_Sk2_1' -> 'M_317_Sk1toM_317_Sk2_1'
 [SVG] Changing 'M_317_Sk1toM_317_Sk2_1' -> 'M_317_Sk1toM_317_Sk2_1' in M317_Sk1-1von1-final.svg
 [